# 01. 시간대별 30분 접근성 및 공식 상권 후보

동대문구 버스정류장·지하철역 334개를 출발지로 두고 08시, 14시, 19시의 30분 대중교통 도달권을 계산합니다. 행정동은 탐색 생활권으로 사용하고, 최종 후보는 서울시 공식 상권 폴리곤으로 정의합니다.

실시간 길찾기 결과가 아니라 공식 운행자료 기반 시간대 평균 정적 모형입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/막시무스'  # 본인 Drive 경로로 수정
%cd $PROJECT_DIR
!pip -q install -r requirements-local-transit.txt
!pip -q install truststore

In [ ]:
from pathlib import Path

required = [
    'data/raw/서울시버스노선별정류소정보_20260902.xlsx',
    'data/raw/tpss_route_section_speedh_2026.08.24-08.30.zip',
    'data/raw/tpss_sta_route_hturn_2026.08.24-08.30.zip',
    'data/raw/kscc_dx_trnsf_path_sum_20260903.zip',
    'data/raw/seoul_metro_timetable_20260616.csv',
    'data/raw/seoul_station_master_20260902.csv',
    'data/external/odsay_origins.csv',
    'data/external/seoul_administrative_dongs_20260701.geojson',
]
missing = [name for name in required if not Path(name).exists()]
for name in required:
    print(('OK  ' if name not in missing else '없음'), name)
if missing:
    raise FileNotFoundError('누락 파일을 data/raw 또는 data/external에 추가한 뒤 다시 실행하세요.')

## 1. 시간대별 네트워크와 대기·환승 보정

버스 최초 대기, 지하철 최초 대기, 환승 추가시간은 각 시간대의 운행자료를 이용해 보정합니다.

In [ ]:
!python -m scripts.build_seoul_transit_network --bus-routes 'data/raw/서울시버스노선별정류소정보_20260902.xlsx' --bus-speeds data/raw/tpss_route_section_speedh_2026.08.24-08.30.zip --metro-timetable data/raw/seoul_metro_timetable_20260616.csv --station-master data/raw/seoul_station_master_20260902.csv --hour 8
!python -m scripts.calibrate_transit_times --bus-operations data/raw/tpss_sta_route_hturn_2026.08.24-08.30.zip --metro-stop-times data/interim/local_transit/metro_stop_times.csv --transfers data/raw/kscc_dx_trnsf_path_sum_20260903.zip --hours 8 14 19

## 2. 08시·14시·19시 30분 도달권 계산

출발지 도보 최대 600m, 환승 도보 최대 250m, 보행속도 1.2m/s를 적용합니다. 각 결과는 시간대별 하위 폴더에 분리 저장합니다.

In [ ]:
import subprocess, sys

base = [
    sys.executable, '-m', 'scripts.build_seoul_transit_network',
    '--bus-routes', 'data/raw/서울시버스노선별정류소정보_20260902.xlsx',
    '--bus-speeds', 'data/raw/tpss_route_section_speedh_2026.08.24-08.30.zip',
    '--metro-timetable', 'data/raw/seoul_metro_timetable_20260616.csv',
    '--station-master', 'data/raw/seoul_station_master_20260902.csv',
]
for hour in (8, 14, 19):
    subprocess.run(base + ['--hour', str(hour)], check=True)
    subprocess.run([
        sys.executable, '-m', 'scripts.static_multimodal_accessibility',
        '--hour', str(hour), '--minutes', '30',
        '--route-waits', f'data/interim/local_transit/calibration/route_waits_{hour:02d}.csv',
        '--output-dir', f'data/processed/local_transit_30min/h{hour:02d}',
    ], check=True)

## 3. 행정동 탐색 생활권

세 시간대의 최소 접근률을 기준으로 25%, 50%, 80% 민감도를 함께 확인합니다. 행정동은 최종 추천 결과가 아니라 공식 상권을 고르기 위한 탐색 범위입니다.

In [ ]:
!python -m scripts.compare_accessibility_periods --threshold 0.25
import pandas as pd
periods = pd.read_csv('data/processed/local_transit_30min/time_period_candidates.csv')
for cutoff in (0.25, 0.50, 0.80):
    count = (periods['minimum_period_ratio'] >= cutoff).sum()
    print(f'세 시간대 공통 {cutoff:.0%} 이상: {count}개 행정동')
display(periods[periods['minimum_period_ratio'] >= 0.25].head(20))

## 4. 공식 상권 폴리곤 후보

상권 경계 400m 안에 30분 내 도달 정류장이 하나 이상 있는 출발지의 비율을 상권 접근률로 사용합니다. 상권이 행정동 경계를 넘는 경우에는 교집합 면적을 보존합니다.

In [ ]:
# 공식 영역·점포·추정매출 ZIP이 없다면 한 번만 실행
from pathlib import Path
area_files = [
    Path('data/raw/commercial_area/commercial_area.zip'),
    Path('data/raw/commercial_area/commercial_store_2025.zip'),
    Path('data/raw/commercial_area/commercial_sales_2025.zip'),
]
if not all(path.exists() for path in area_files):
    subprocess.run([sys.executable, '-m', 'scripts.download_seoul_commercial_area_sources'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.build_commercial_area_accessibility'], check=True)

areas = pd.read_csv('data/processed/commercial_area_accessibility/commercial_area_candidates_25pct.csv')
print(f'공식 상권 25% 후보: {len(areas)}개')
display(areas[[
    'area_name', 'area_type_name', 'primary_admin_name',
    'ratio_08', 'ratio_14', 'ratio_19', 'minimum_period_ratio',
    'store_count_total', 'sales_amount_total',
]].head(20))

## 결과 확인

- 행정동 결과: `data/processed/local_transit_30min/time_period_candidates.csv`
- 공식 상권 후보: `data/processed/commercial_area_accessibility/commercial_area_candidates_25pct.csv`
- 지도 파일: `data/processed/commercial_area_accessibility/commercial_area_candidates_25pct.geojson`

이 노트북의 결과는 시간대 평균 접근성 후보 생성 결과입니다. 실서비스 또는 개별 사용자 추천에서는 길찾기 API로 최종 경로를 다시 확인합니다.